# Day 3 — Semantic Retrieval

Goal: given a free-text trial idea, return the most similar historical trials + their outcomes.
Saves `../src/trials.faiss` and `../src/trials_meta.parquet` (the Day 4 API loads both).

Run top to bottom with **Shift+Enter**. Pick your **Python** kernel if asked. Read curriculum section 8 first.

Note: the first cell downloads a ~80MB embedding model the first time — give it a minute.

## Step 1 — load data + the embedding model

In [1]:
import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

df = pd.read_csv("../data/trials_features.csv").dropna(subset=["brief_summary"]).reset_index(drop=True)
df["label"] = (df["overall_status"].str.upper() == "COMPLETED").astype(int)
print("Trials with a summary:", len(df))

model = SentenceTransformer("all-MiniLM-L6-v2")   # fast, strong generalist
print("Model loaded.")

Trials with a summary: 155335


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded.


## Step 2 — embed all trial summaries

This is the slow cell. ~1–3 min for tens of thousands of trials on CPU. `normalize_embeddings=True` lets us use inner product as cosine similarity.

In [2]:
emb = model.encode(
    df["brief_summary"].tolist(),
    show_progress_bar=True, convert_to_numpy=True, normalize_embeddings=True
)
print("Embedding matrix:", emb.shape)   # (n_trials, 384)

Batches:   0%|          | 0/4855 [00:00<?, ?it/s]

Embedding matrix: (155335, 384)


## Step 3 — build the FAISS index and save

In [3]:
index = faiss.IndexFlatIP(emb.shape[1])   # inner product on normalized vectors = cosine
index.add(emb)
faiss.write_index(index, "../src/trials.faiss")
df.to_parquet("../src/trials_meta.parquet")
print("Indexed", index.ntotal, "trials -> ../src/trials.faiss")

Indexed 155335 trials -> ../src/trials.faiss


## Step 4 — the search function

In [4]:
def search(query, k=5):
    q = model.encode([query], normalize_embeddings=True)
    scores, idx = index.search(q, k)
    out = df.iloc[idx[0]][["nct_id", "phase", "overall_status", "label", "brief_summary"]].copy()
    out["similarity"] = scores[0]
    return out

res = search("Phase 2 trial of a GLP-1 agonist for type 2 diabetes in adults")
res

,nct_id,phase,overall_status,label,brief_summary,similarity
42868,NCT00859079,PHASE4,COMPLETED,1,The aim of the investigators study was to comp...,0.806190
96091,NCT02518945,PHASE3,COMPLETED,1,"This is a single center, prospective, randomiz...",0.803098
79753,NCT01931982,PHASE4,COMPLETED,1,The purpose of the study is to determine if a ...,0.777447
142651,NCT05297045,PHASE2,TERMINATED,0,This is a phase 2 study designed to evaluate t...,0.765370
25923,NCT00460941,PHASE2,COMPLETED,1,This 4 arm study will evaluate the tolerabilit...,0.763842


## Step 5 — sanity-check retrieval on a few queries

Eyeball these: do the returned trials actually match the query's topic? That's your evidence the retrieval works.

In [6]:
for q in [
    "immunotherapy for advanced melanoma",
    "randomized trial of a statin to lower cholesterol",
    "vaccine study in healthy adult volunteers",
]:
    print("="*80)
    print("QUERY:", q)
    r = search(q, k=3)
    for _, row in r.iterrows():
        print(f"  [{row['similarity']:.2f}] {row['nct_id']} ({row['overall_status']}): {row['brief_summary'][:120]}...")
    print(f"  -> historical completion rate of these {len(r)}: {r['label'].mean():.2f}")

QUERY: immunotherapy for advanced melanoma
  [0.85] NCT04007588 (WITHDRAWN): This research study is studying different immunotherapy regimens as a possible treatment for stage III or IV resectable ...
  [0.84] NCT03338777 (TERMINATED): Safety evaluation of combined immunogene therapy in patients with advanced melanoma....
  [0.83] NCT01026051 (TERMINATED): The clinical trial is evaluating a multi-component active immunotherapy designed to stimulate an immune reaction to spec...
  -> historical completion rate of these 3: 0.00
QUERY: randomized trial of a statin to lower cholesterol
  [0.82] NCT00299169 (TERMINATED): Patients who are intolerant of statins in routine practice, but who lack objective evidence of significant harm, will be...
  [0.78] NCT00442325 (COMPLETED): European physicians tend to always use the lowest dose of statins to initiate therapy even in subjects who require large...
  [0.78] NCT00442845 (COMPLETED): Physicians tend to always use the lowest dose of statins to 

## Step 6 (optional, high-signal) — generalist vs biomedical embedder bake-off

Research finding: on short clinical text, general models often beat specialized biomedical ones. Test it and report which won — a memorable interview point. Skip if short on time (downloads another model).

In [7]:
# OPTIONAL — comment out if short on time.
bio = SentenceTransformer("pritamdeka/S-PubMedBert-MS-MARCO")
sample = df.sample(min(3000, len(df)), random_state=1).reset_index(drop=True)
bio_emb = bio.encode(sample["brief_summary"].tolist(), convert_to_numpy=True,
                     normalize_embeddings=True, show_progress_bar=True)
bio_index = faiss.IndexFlatIP(bio_emb.shape[1]); bio_index.add(bio_emb)

def bio_search(query, k=3):
    q = bio.encode([query], normalize_embeddings=True)
    s, i = bio_index.search(q, k)
    return sample.iloc[i[0]][["nct_id", "brief_summary"]].assign(similarity=s[0])

print("Biomedical model results for 'immunotherapy for advanced melanoma':")
for _, row in bio_search("immunotherapy for advanced melanoma").iterrows():
    print(f"  [{row['similarity']:.2f}] {row['brief_summary'][:120]}...")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/4.56k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/666 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/461k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/94 [00:00<?, ?it/s]

Biomedical model results for 'immunotherapy for advanced melanoma':
  [0.93] RATIONALE: Immunotoxins can locate tumor cells and kill them without harming normal cells. Immunotoxin therapy may be an...
  [0.92] Open label, phase Ib study of intratumoral tilsotolimod in combination with intratumoral ipilimumab and intravenous nivo...
  [0.92] The investigators propose a trial to evaluate if the addition of ipilimumab to nivolumab after primary or acquired resis...


In [8]:
import os
print("faiss index exists:", os.path.exists("../src/trials.faiss"))
print("metadata exists:", os.path.exists("../src/trials_meta.parquet"))

faiss index exists: True
metadata exists: True
